# GitHub PR Review Agent using NKit & Gemini

This notebook demonstrates building a GitHub Pull Request Review Agent that fetches the actual diff of a PR and reviews it to find the two biggest architectural or code-quality issues, with a specific focus on SOLID principles.

In [ ]:
import os
import sys
import logging
import urllib.request
import urllib.error
from IPython.display import display, Markdown

# 1. Unload globally installed 'nkit' to prevent shadowing local changes
if 'nkit' in sys.modules:
    del sys.modules['nkit']
for key in list(sys.modules.keys()):
    if key.startswith('nkit.'):
        del sys.modules[key]

# 2. Find the robust path to the project root containing the NKit source code
current_dir = os.getcwd()
while current_dir and not os.path.isdir(os.path.join(current_dir, 'NKit')):
    parent = os.path.dirname(current_dir)
    if parent == current_dir: 
        break
    current_dir = parent

if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"Using NKit repository at: {current_dir}")

# Import NKit dynamically from local rather than PyPI
from NKit import Agent, Tool, LiveObserver
from NKit.llms import GeminiLLM
from NKit.safety import SafetyGate
from NKit.audit import WhyLog

# Suppress verbose HTTP logs if desired
logging.getLogger("urllib3").setLevel(logging.WARNING)

## Configuration
Provide your API Keys.

In [ ]:
# Required for the LLM Provider
if "GEMINI_API_KEY" not in os.environ:
    print("⚠️ Please set GEMINI_API_KEY to continue.")
    # os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Optional but highly recommended for GitHub API Rate Limits
if "GITHUB_TOKEN" not in os.environ:
    print("⚠️ Consider setting GITHUB_TOKEN to avoid GitHub API rate limits. (Public repos only without it)")
    # os.environ["GITHUB_TOKEN"] = "your_github_token_here"

## Define GitHub PR Tools
The agent needs the ability to fetch the PR diff and information so that it can evaluate the code effectively.

In [ ]:
def get_pr_diff(owner: str, repo: str, pr_number: int) -> str:
    """
    Fetches the raw diff of a GitHub Pull Request so you can analyze the code changes.
    
    Args:
        owner: The owner of the GitHub repository (e.g., 'nkit-ai')
        repo: The name of the repository (e.g., 'nkit')
        pr_number: The pull request number
    """
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"
    req = urllib.request.Request(url)
    
    # Requesting the v3.diff format gets us the raw Git patch block of the exact code changes!
    req.add_header("Accept", "application/vnd.github.v3.diff")
    req.add_header("User-Agent", "NKit-PR-Agent")
    
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        req.add_header("Authorization", f"Bearer {token}")
        
    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            diff_data = response.read().decode('utf-8')
            if len(diff_data) == 0:
                return "The PR diff is unexpectedly empty. Is it a valid PR?"
            
            # Optionally truncate massively excessive diffs so we don't blow up context limits
            if len(diff_data) > 30000:
                return diff_data[:30000] + "\n\n[DIFF TRUNCATED TO AVOID BLOWING UP CONTEXT LIMITS]"
            return diff_data
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return f"PR not found or unauthorized: HTTP 404. Make sure repository {owner}/{repo}/pull/{pr_number} is public or GITHUB_TOKEN is set."
        return f"Failed to fetch PR diff: HTTP {e.code} - {e.reason}"
    except Exception as e:
        return f"Failed to fetch PR diff: {str(e)}"

# Wrap it into an NKit Tool
pr_diff_tool = Tool(
    name="get_pr_diff",
    func=get_pr_diff,
    desc="Fetches the raw code diff (git patch) for a GitHub Pull Request. Provide the owner, repo name, and numeric PR pull ID."
)

## Initialize NKit Components

In [ ]:
# Note: the model determines the upper bounds of our context limit.
# Gemini 2.5 Flash has a large context window, so it works really well for massive log parsing!
llm = GeminiLLM(model="gemini-2.5-flash")
observer = LiveObserver()

# Establish standard auditing trail
os.makedirs("./logs", exist_ok=True)
audit_log = WhyLog(path="./logs/github_agent_audit.jsonl")

agent = Agent(
    llm=llm,
    observer=observer,
    why_log=audit_log
)

# CORRECTION: We need to use `agent.registry.register` or `agent.add_tool` to inject it!
agent.registry.register(pr_diff_tool)
    
# Dynamic insight streaming into our Notebook
@observer.on("agent.reasoning")
def print_thoughts(event):
    print(f"\n[THOUGHT]: {event.get('thought', '')}")

@observer.on("tool.before")
def print_action(event):
    print(f"► EXECUTING: {event['tool_name']} | WHY: {event.get('why', 'No reason')}")

## Start Review Process

When you run this cell, it will prompt you for the specific Pull Request you want to analyze!

In [ ]:
print("=== GitHub Agent Setup ===")
repo_owner = input("Enter the GitHub repository owner (e.g., 'microsoft'): ").strip()
repo_name = input("Enter the repository name (e.g., 'vscode'): ").strip()
pr_num_str = input("Enter the PR number (e.g., '200000'): ").strip()

try:
    pr_num = int(pr_num_str)
except ValueError:
    print("\n[Error] Invalid PR number. Please run the cell again and provide a valid number.")
    pr_num = None

if pr_num:
    prompt = f"""
You are a Senior Staff Software Engineer and an expert in Object-Oriented Design and SOLID principles.

Currently, you are reviewing Pull Request #{pr_num} for the repository '{repo_owner}/{repo_name}'.
1. Use the 'get_pr_diff' tool to fetch the exact code changes introduced in this PR.
2. Carefully analyze the code differences for structural flaws, particularly violations of SOLID principles (Single Responsibility, Open-Closed, Liskov Substitution, Interface Segregation, Dependency Inversion).
3. Identify the TWO biggest structural or SOLID issues out of the entire patch set.
4. Finally, return a beautiful Markdown response that explains:
   - Issue 1: Explain the code segment, why it violates SOLID principles, and propose a specific refactor.
   - Issue 2: Do the exact same thing.
   - Note: If the code is perfect and contains zero issues, applaud the author and state there are no SOLID violations.
"""
    print(f"\n[Starting Review] Analyzing {repo_owner}/{repo_name} PR #{pr_num}...")
    
    try:
        # JUPYTER FIX: execute correctly within the event loop!
        result = await agent.run_async(prompt)
        print("\n\n" + "="*40 + " [FINAL REVIEW] " + "="*40)
        display(Markdown(result))
    except Exception as e:
        print(f"Execution Error: {e}")

## CodeACT: Pure Autonomous Runtime Tools
Instead of spoon-feeding the `get_pr_diff` tool to the agent, we will spin up a fresh agent armed ONLY with CodeACT (`execute_python` and `inject_dynamic_tool`). We will simply instruct the agent to fetch a PR diff and review it. To complete this mission, the Agent will literally code and inject its own python script into its tool registry!

In [ ]:
# Clear the previous observer and instantiate a fresh God Agent
# Note: The Agent inherits the dynamic CodeACT tools automatically from the registry update we just pushed to the NKit core!
dynamic_agent = Agent(llm=llm, observer=observer, why_log=audit_log)

print("=== Dynamic CodeACT Setup ===")
repo_owner = input("Enter the GitHub repository owner (e.g., 'microsoft'): ").strip()
repo_name = input("Enter the repository name (e.g., 'vscode'): ").strip()
pr_num_str = input("Enter the PR number (e.g., '200000'): ").strip()

try:
    pr_num = int(pr_num_str)
except ValueError:
    print("\n[Error] Invalid PR number.")
    pr_num = None

if pr_num:
    prompt = f"""
You are an autonomous AI Agent.
Your objective is to fetch Pull Request #{pr_num} for the repository '{repo_owner}/{repo_name}' and explain the two biggest violations of SOLID principles.

IMPORTANT CODEACT INSTRUCTIONS:
You do NOT currently have a pre-built tool to fetch GitHub PRs! You must build one yourself on the fly.
1. Use your 'inject_dynamic_tool' tool to write a python function named 'fetch_pr_code'. 
   The python script you write must define this function and use urllib or requests to fetch: https://api.github.com/repos/{repo_owner}/{repo_name}/pulls/{pr_num} 
   It MUST pass the header {{"Accept": "application/vnd.github.v3.diff"}} in the request.
2. Once 'fetch_pr_code' tool is injected globally successfully, you MUST actually call your newly generated 'fetch_pr_code' tool to pull the patch set.
3. Review the code exactly as instructed and return a beautifully formatted Markdown response.
"""
    print(f"\n[CodeACT Agent Starting] Analyzing {repo_owner}/{repo_name} PR #{pr_num} autonomously...")
    
    try:
        result = await dynamic_agent.run_async(prompt)
        print("\n\n" + "="*40 + " [FINAL REVIEW] " + "="*40)
        display(Markdown(result))
    except Exception as e:
        print(f"Execution Error: {e}")